# Losses: Metric, Contrastive, and Structured Vision (C03 Topic 08)

**Citations:** Oord et al. (2018 InfoNCE); Zhai et al. (2023 SigLIP); Deng et al. (2019 ArcFace); Milletari et al. (2016 Soft-Dice); Rezatofighi et al. (2019 GIoU).

### Purpose & Topic Bridges
Standard point-wise losses ignore embedding geometry, mask overlap, and spatial enclosure. This notebook implements InfoNCE contrastive mutual information estimation, SigLIP pairwise sigmoid BCE for distributed pretraining, ArcFace additive angular margin on hyperspheres, Soft-Dice loss for imbalanced segmentation masks, and Generalized IoU (GIoU) for bounding box regression.

## 2. Mathematical Foundations & Governing Equations

1. **InfoNCE:** $\mathcal{L} = -\log \frac{\exp(q \cdot k_+ / \tau)}{\sum_j \exp(q \cdot k_j / \tau)}$, maximizing lower bound on mutual information on the unit hypersphere.
2. **SigLIP:** $\mathcal{L} = \frac{1}{B} \sum_{i,j} \log(1 + e^{-y_{ij} (t (q_i \cdot k_j) + b)})$, replacing global softmax with pairwise sigmoid BCE to eliminate distributed all-gather barriers.
3. **ArcFace:** $\cos(\theta_y + m) = \cos \theta_y \cos m - \sin \theta_y \sin m$, enforcing additive angular margin $m$ directly on the geodesic hypersphere.
4. **Soft-Dice:** $\mathcal{L} = 1 - \frac{2 \sum p_i y_i + \epsilon}{\sum p_i^2 + \sum y_i^2 + \epsilon}$, providing scale-invariant optimization for extreme foreground-background mask imbalance.
5. **GIoU:** $\text{GIoU} = \text{IoU} - \frac{|C \setminus (A \cup B)|}{|C|}$, adding an enclosing convex hull penalty $C$ to guarantee non-zero gradients on disjoint boxes.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/08_losses_metric_contrastive_and_structured_vision.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class MetricVisionUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def infonce_loss(self, q, k, tau=0.07):
        qn, kn = F.normalize(q, p=2, dim=-1), F.normalize(k, p=2, dim=-1)
        logits = torch.matmul(qn, kn.t()) / tau
        return F.cross_entropy(logits, torch.arange(q.shape[0], device=q.device))

    def siglip_loss(self, q, k, scale=10.0, bias=-10.0):
        qn, kn = F.normalize(q, p=2, dim=-1), F.normalize(k, p=2, dim=-1)
        logits = torch.matmul(qn, kn.t()) * scale + bias
        labels = 2.0 * torch.eye(q.shape[0], device=q.device) - 1.0
        return F.softplus(-labels * logits).mean()

    def arcface_loss(self, emb, w, targets, s=30.0, m=0.5, eps=1e-7):
        en, wn = F.normalize(emb, p=2, dim=-1), F.normalize(w, p=2, dim=-1)
        cos_t = torch.matmul(en, wn.t()).clamp(-1.0 + eps, 1.0 - eps)
        sin_t = torch.sqrt((1.0 - cos_t.pow(2)).clamp(min=eps))
        cos_tm = cos_t * math.cos(m) - sin_t * math.sin(m)
        oh = F.one_hot(targets, num_classes=w.shape[0]).float()
        return F.cross_entropy(s * (oh * cos_tm + (1.0 - oh) * cos_t), targets)

    def soft_dice_loss(self, probs, targets, eps=1e-6):
        pf, tf = probs.reshape(-1), targets.reshape(-1)
        return 1.0 - (2.0 * (pf * tf).sum() + eps) / (pf.pow(2).sum() + tf.pow(2).sum() + eps)

    def giou_loss(self, b1, b2, eps=1e-7):
        x1_i, y1_i = torch.max(b1[:, 0], b2[:, 0]), torch.max(b1[:, 1], b2[:, 1])
        x2_i, y2_i = torch.min(b1[:, 2], b2[:, 2]), torch.min(b1[:, 3], b2[:, 3])
        inter = (x2_i - x1_i).clamp(min=0.0) * (y2_i - y1_i).clamp(min=0.0)
        a1 = (b1[:, 2] - b1[:, 0]).clamp(min=0.0) * (b1[:, 3] - b1[:, 1]).clamp(min=0.0)
        a2 = (b2[:, 2] - b2[:, 0]).clamp(min=0.0) * (b2[:, 3] - b2[:, 1]).clamp(min=0.0)
        union = a1 + a2 - inter + eps
        x1_c, y1_c = torch.min(b1[:, 0], b2[:, 0]), torch.min(b1[:, 1], b2[:, 1])
        x2_c, y2_c = torch.max(b1[:, 2], b2[:, 2]), torch.max(b1[:, 3], b2[:, 3])
        c_area = (x2_c - x1_c).clamp(min=0.0) * (y2_c - y1_c).clamp(min=0.0) + eps
        return (1.0 - (inter / union - (c_area - union) / c_area)).mean()

model = MetricVisionUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('MetricVisionUnifiedModule initialized successfully.')

In [ ]:
q, k, tau = weights['q_repr'], weights['k_repr'], float(weights['tau_val'][0])
sc, sb = float(weights['siglip_scale'][0]), float(weights['siglip_bias'][0])
a_emb, a_w, a_t = weights['arc_emb'], weights['arc_weights'], weights['arc_targets'].to(torch.int64)
as_s, as_m = float(weights['arc_s'][0]), float(weights['arc_m'][0])
d_p, d_t = weights['dice_probs'], weights['dice_targets']
bx1, bx2 = weights['boxes1'], weights['boxes2']

err_info = (model.infonce_loss(q, k, tau) - weights['loss_infonce'][0]).abs().item()
assert err_info < 1e-5, f'InfoNCE parity error: {err_info:.2e}'

err_sig = (model.siglip_loss(q, k, sc, sb) - weights['loss_siglip'][0]).abs().item()
assert err_sig < 1e-5, f'SigLIP parity error: {err_sig:.2e}'

err_arc = (model.arcface_loss(a_emb, a_w, a_t, as_s, as_m) - weights['loss_arcface'][0]).abs().item()
assert err_arc < 1e-5, f'ArcFace parity error: {err_arc:.2e}'

err_dice = (model.soft_dice_loss(d_p, d_t) - weights['loss_dice'][0]).abs().item()
assert err_dice < 1e-5, f'Soft-Dice parity error: {err_dice:.2e}'

err_giou = (model.giou_loss(bx1, bx2) - weights['loss_giou'][0]).abs().item()
assert err_giou < 1e-5, f'GIoU parity error: {err_giou:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 6 invariants verified! InfoNCE: {err_info:.2e}, SigLIP: {err_sig:.2e}, ArcFace: {err_arc:.2e}, Dice: {err_dice:.2e}, GIoU: {err_giou:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'InfoNCE Loss': lambda: model.infonce_loss(q, k, tau),
    'SigLIP Loss': lambda: model.siglip_loss(q, k, sc, sb),
    'ArcFace Loss': lambda: model.arcface_loss(a_emb, a_w, a_t, as_s, as_m),
    'Soft-Dice Loss': lambda: model.soft_dice_loss(d_p, d_t),
    'GIoU Loss': lambda: model.giou_loss(bx1, bx2),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Zero-Overlap Gradient Death in Vanilla IoU vs Non-Zero GIoU Convex Hull Gradient
# Two disjoint bounding boxes: Box A at [0, 0, 10, 10], Box B at [50, 50, 60, 60]
b_pred = torch.tensor([[0.0, 0.0, 10.0, 10.0]], requires_grad=True, device=device)
b_gt = torch.tensor([[50.0, 50.0, 60.0, 60.0]], device=device)

# 1. Vanilla IoU Loss: 1 - IoU
inter_vanilla = (torch.min(b_pred[:, 2], b_gt[:, 2]) - torch.max(b_pred[:, 0], b_gt[:, 0])).clamp(min=0.0) * \
                (torch.min(b_pred[:, 3], b_gt[:, 3]) - torch.max(b_pred[:, 1], b_gt[:, 1])).clamp(min=0.0)
loss_vanilla_iou = 1.0 - (inter_vanilla / 200.0) # union=200
loss_vanilla_iou.backward()
grad_vanilla = b_pred.grad.abs().sum().item()
print(f'[FAILURE DEMONSTRATED] Disjoint boxes under Vanilla IoU produce ZERO gradient: {grad_vanilla:.4f}')
assert grad_vanilla == 0.0, f'Expected zero gradient on non-overlapping boxes, got {grad_vanilla}'

# 2. Generalized IoU Loss (GIoU): Enclosing box convex hull maintains healthy non-zero gradient force
b_pred_giou = torch.tensor([[0.0, 0.0, 10.0, 10.0]], requires_grad=True, device=device)
loss_giou = model.giou_loss(b_pred_giou, b_gt)
loss_giou.backward()
grad_giou = b_pred_giou.grad.abs().sum().item()
print(f'[STABILIZED] GIoU produces active gradient pulling disjoint boxes together: {grad_giou:.4f}')
assert grad_giou > 1e-4, f'Expected non-zero gradient from GIoU, got {grad_giou}'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Metric vision losses and linear anchor verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Decoupled Contrastive Scaling:** SigLIP replaces global cross-GPU softmax synchronization with independent pairwise sigmoid BCE, eliminating communication bottlenecks in web-scale pretraining.
2. **Geodesic Angular Bounds:** ArcFace's additive angular margin $\cos(\theta + m)$ compresses intra-class features and expands inter-class boundaries on the hypersphere.
3. **Convex Hull Continuity:** GIoU resolves the fundamental discontinuity of standard IoU on disjoint boxes, ensuring stable gradient propagation across dense object detectors.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 07_losses_regression_classification_and_barron_robust](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/07_losses_regression_classification_and_barron_robust.ipynb)